In [1]:
from dotenv import load_dotenv
import os
from huggingface_hub import login

load_dotenv()
hf_token = os.environ["HUGGINGFACE_HUB_TOKEN"]
login(token=hf_token)
print(len(hf_token) != 0)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True


In [2]:
from pathlib import Path

with open("data-rag.jsonl","w", encoding="utf-8") as out:
    for path in Path("data").rglob("*"):
        if path.name == "data.jsonl":
            with path.open("r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        out.write(line.rstrip("\n") + "\n")

In [4]:
import json

data = []
with open('data-rag.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

def convert_data(item):
    stripped_prompt = " ".join(item["prompt"].split(" ")[1:])
    conversation =  [{
                "role": "user",
                "content": stripped_prompt
            }, {
                "role": "assistant",
                "content": item["completion"]
            }]
    text = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {
        "text": text
    }


data = [convert_data(item) for item in data]
    
with open('data-rag.jsonl', 'w', encoding='utf-8') as f:
     for item in data:
        f.write(json.dumps(item) + '\n')

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_name = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto").to("cuda")

# LoRA config
lora_config = LoraConfig(
    r=4,                      
    lora_alpha=8,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.tie_weights()
model.print_trainable_parameters()

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.42it/s]


trainable params: 6,078,464 || all params: 3,218,828,288 || trainable%: 0.1888


In [3]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="data-rag.jsonl", split="train")

In [4]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="llama323-finetuned",
        per_device_train_batch_size=4,   # small batch size for Colab GPU
        gradient_accumulation_steps=4,   # accumulate gradients to simulate larger batch
        num_train_epochs=6,
        learning_rate=5e-5,
        logging_steps=50,
        save_strategy="epoch",
        warmup_ratio=0.05,
        weight_decay=0.01,
        lr_scheduler_type="cosine"
    ),
)

trainer.train()
model.save_pretrained("llama323-finetuned")

Truncating train dataset: 100%|██████████| 722/722 [00:00<00:00, 103943.42 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss
50,2.820700
100,1.452100
150,1.310700
200,1.248900
250,1.202700
